In [11]:
from datasets import load_dataset
import os
from itertools import islice
import random
import unicodedata
import re

In [2]:
os.makedirs("./data", exist_ok=True)

Load Hindi Dataset

In [3]:
hindi_dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files="data/hi-1.txt",   
    split="train",
    streaming=True
)

with open("./data/hindi_dataset.txt", "w", encoding="utf-8") as f:
    for sample in islice(hindi_dataset, 50000):
        
        text = sample["text"] if isinstance(sample, dict) else sample
        text = text.strip()

        if text:
            f.write(text + "\n")

print("Saved Hindi Dataset!")

Saved Hindi Dataset!


Load English Dataset

In [4]:
english_dataset = load_dataset("wikitext", "wikitext-103-v1")

count = 0
MAX_LINES = 25000

with open("./data/english_dataset.txt", "w", encoding="utf-8") as f:
    for sample in english_dataset["train"]:  
        
        text = sample["text"] if isinstance(sample, dict) else sample
        text = text.strip()

        if text:
            f.write(text + "\n")
            count += 1
        
        if count >= MAX_LINES:
            break

print("Saved English dataset!")

Saved English dataset!


Pre-process and clean data

In [12]:
def clean_text(text, lang):
    text = text.strip()

    if not text:
        return None

    if text.startswith("=") and text.endswith("="):
        return None

    if "<unk>" in text:
        return None

    text = re.sub(r"\s*@-@\s*", "-", text)   # re @-@ enjoy → re-enjoy
    text = re.sub(r"\s*@,@\s*", ",", text)   # 10 @,@ 000 → 10,000

    text = re.sub(r"\s+", " ", text)

    # normalize unicode (important for Hindi)
    text = unicodedata.normalize("NFKC", text)

    # lowercase only English
    if lang == "EN":
        text = text.lower()

    if len(text) < 10:
        return None

    return f"[{lang}] {text}"

In [13]:
cleaned = []

# Hindi
with open("./data/hindi_dataset.txt", "r", encoding="utf-8") as f:
    for line in f:
        c = clean_text(line, "HI")
        if c:
            cleaned.append(c)

# English
with open("./data/english_dataset.txt", "r", encoding="utf-8") as f:
    for line in f:
        c = clean_text(line, "EN")
        if c:
            cleaned.append(c)

Random shuffle and save data

In [14]:
cleaned = list(set(cleaned))

random.shuffle(cleaned)

In [15]:
with open("./data/final_dataset.txt", "w", encoding="utf-8") as f:
    for line in cleaned:
        f.write(line + "\n")